# 第7周-Day3：任务编排与工作流

> 背景：Jason 已完成 W1-W6 大模型基础，W7 前两天分别学了数字员工行为设计和记忆系统。
>
> 今天的主题是：**把 Agent 从单点回答，推进到可持续执行任务的工作流系统**。

## 今日目标
- 理解为什么数字员工必须有任务编排
- 区分线性流程、分支流程、并行流程、事件驱动流程和 Supervisor 编排
- 掌握 Cron、延迟任务、周期任务的设计方式
- 理解 DAG / 状态机 / checkpoint 在运行时中的作用
- 能把任务编排映射到 LangChat、OpenClaw、商管、会员、物业场景

## 往期回顾：W7 Day1-Day2

Day1 解决的是“**这个数字员工是谁、怎么说话、能不能做**”。

Day2 解决的是“**它记得什么、怎么检索、怎么跨会话保持连续性**”。

今天要解决的是第三个问题：**它如何把一串动作按顺序、按条件、按时间真正执行完**。

| 能力层 | 代表问题 | 结果 |
|---|---|---|
| 行为设计 | 它是谁 | 有人格、有边界 |
| 记忆系统 | 它记得什么 | 能持续对话 |
| 任务编排 | 它怎么做事 | 能完成流程 |

In [ ]:
# matplotlib 中文字体配置（必须是第一段代码）
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np
import math

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)

In [ ]:
# 学习进度条：W1-W6 已完成，W7 进行中，共 18 周
weeks = [f"W{i}" for i in range(1, 19)]
status = ["done"] * 6 + ["doing"] + ["todo"] * 11
colors = {"done": "#5CB85C", "doing": "#F0AD4E", "todo": "#E6E6E6"}

fig, ax = plt.subplots(figsize=(15, 2.4))
for idx, (week, st) in enumerate(zip(weeks, status)):
    ax.barh(0, 1, left=idx, height=0.62, color=colors[st], edgecolor="white")
    ax.text(idx + 0.5, 0, week, ha="center", va="center", fontsize=10, color="#222")

ax.set_xlim(0, 18)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title("18周学习进度：W1-W6 已完成，W7 进行中", fontsize=14, pad=14)
ax.text(0, -0.9, "已完成 6/18 | 进行中 1/18 | 剩余 11 周", fontsize=11)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
plt.tight_layout()
plt.show()

## 1. 为什么数字员工必须做任务编排

聊天机器人擅长回答问题，但企业里真正费时间的，通常不是回答，而是**把一串动作做完**。

一个典型任务可能是：识别意图 → 查询数据 → 判断是否要走审批 → 通知用户 → 记录审计 → 安排下次跟进。

如果没有编排，Agent 就会变成“会想但不会做”的状态；如果编排过度，又会变成僵硬的脚本。工作流的价值，就是把两者之间的边界定住。

## 2. 工作流不是一条链，而是一张图

很多人把 workflow 想成“步骤 1、步骤 2、步骤 3”的直线。那只适合简单任务。

真实业务里，动作通常会分叉：
- 审批通过，进入执行
- 审批拒绝，进入人工处理
- 数据缺失，进入补参
- 风险过高，进入拦截

所以更准确的抽象不是链，而是 **DAG / 状态机 / 事件图**。

### 直观类比
- 线性流程：流水线
- 分支流程：闸机
- 并行流程：几个人同时分工
- 事件驱动：有人发消息，系统再响应
- Supervisor：一个主管分派任务给多个专员

In [ ]:
import matplotlib.pyplot as plt

patterns = ["线性", "分支", "并行", "事件驱动", "Supervisor"]
latency = [2, 3, 2, 4, 3]
flexibility = [1, 3, 4, 5, 5]
traceability = [5, 4, 3, 4, 5]

x = np.arange(len(patterns))
width = 0.24

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x - width, latency, width, label="延迟(越低越好)", color="#5B8FF9")
ax.bar(x, flexibility, width, label="灵活性", color="#61DDAA")
ax.bar(x + width, traceability, width, label="可追踪性", color="#F6BD16")

ax.set_xticks(x)
ax.set_xticklabels(patterns)
ax.set_ylim(0, 6)
ax.set_ylabel("相对评分（1-5）")
ax.set_title("常见编排模式的工程权衡")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.15))
plt.tight_layout()
plt.show()

## 3. 线性流程：最适合确定性强的任务

线性流程的特点很简单：输入进去，按固定顺序往下走。

适合的场景包括：
- 先分类，再回答
- 先生成草稿，再审校，再发布
- 先拉取数据，再计算，再渲染图表

它的优点是简单、稳定、好测试；缺点是遇到不确定性就容易卡死，因为它没有“换路”的能力。

In [ ]:
def linear_workflow(order_id):
    steps = []
    steps.append(f"1. 接收订单 {order_id}")
    steps.append("2. 查询订单状态")
    steps.append("3. 判断是否满足退款条件")
    steps.append("4. 生成结果并通知用户")
    return steps

for item in linear_workflow('A1024'):
    print(item)

## 4. 分支流程：把不确定性变成可控的路由

分支不是“模型自由发挥”，而是**把条件判断写成显式路由**。

企业里常见的分支条件：
- 金额是否超过阈值
- 用户是否有权限
- 数据是否完整
- 是否命中风控规则

分支的关键是：判断条件必须清楚，不能让模型自己猜。模型负责理解语义，路由规则负责决定走哪条路。

In [ ]:
def route_task(request):
    if '退款' in request:
        return 'refund_flow'
    if '改地址' in request:
        return 'address_change_flow'
    if '报表' in request:
        return 'report_flow'
    return 'human_review'

samples = ['我要退款', '帮我改地址', '生成月报', '这事比较复杂']
for s in samples:
    print(s, '->', route_task(s))

## 5. 并行流程：把等待时间切开

并行的价值不是“更高级”，而是**把互不依赖的等待拆开**。

例如一个月度经营分析任务，可以同时做：
- 拉销售数据
- 拉会员数据
- 拉商管数据
- 拉物业数据

最后再汇总。这样做的好处是缩短总时长，也让每条链路更清晰。

但并行也有代价：要处理结果合并、失败重试和一致性问题。

In [ ]:
tasks = {
    'sales': 1.8,
    'membership': 1.2,
    'property': 1.5,
    'commercial': 1.7,
}

serial_time = sum(tasks.values())
parallel_time = max(tasks.values())
print(f"串行耗时约 {serial_time:.1f}s")
print(f"并行耗时约 {parallel_time:.1f}s")
print(f"节省 {serial_time - parallel_time:.1f}s")

## 6. 事件驱动：让系统对外部变化做响应

事件驱动流程不是“我现在就做”，而是“**一旦发生，就触发**”。

例子：
- 订单支付成功后触发发货
- 会员等级变更后触发权益刷新
- 工单超时后触发提醒
- 价格波动后触发重新计算

这类流程特别适合 OpenClaw 这种 Orchestrator 场景，因为它天然面对的是大量异步事件。

## 7. Supervisor 模式：一个主管协调多个专员

Supervisor 编排的核心是：有一个中央节点负责决定谁做什么，专员只做自己擅长的事。

这比“一个 Agent 什么都干”更像企业组织结构。

### 适合的情况
- 研究、写作、审校、发布要分工
- 数据、规则、沟通、审批不能混在一个脑子里
- 任务需要清晰的责任边界和日志追踪

### 不适合的情况
- 小任务，单节点就够
- 路由比执行还复杂，反而增加管理成本

In [ ]:
specialists = {
    'researcher': '搜集资料和证据',
    'planner': '拆解任务和安排顺序',
    'executor': '调用工具执行动作',
    'reviewer': '检查一致性和风险',
}

for name, duty in specialists.items():
    print(f'{name}: {duty}')

## 8. Cron、延迟任务、周期任务

企业里很多工作不是“马上做完”，而是“到了时间再做”。

Cron 适合固定时刻触发，比如每天 6 点推送；
延迟任务适合“过 10 分钟提醒我”；
周期任务适合“每隔 30 分钟同步一次状态”。

### 关键原则
- 用墙钟时间表达，不要靠人脑记
- 写明时区
- 任务要可重入，避免重复触发造成副作用
- 高风险动作最好先做 dry-run，再做正式执行

In [ ]:
schedule = [
    {'name': 'daily_push', 'type': 'cron', 'expr': '0 6 * * *', 'tz': 'Asia/Shanghai'},
    {'name': 'review_after_10min', 'type': 'delay', 'delay_minutes': 10},
    {'name': 'sync_every_30min', 'type': 'interval', 'every_minutes': 30},
]

for item in schedule:
    print(item)

## 9. DAG、状态机和 checkpoint

如果把 workflow 看成运行时结构，那么三个词最重要：

- **DAG**：谁依赖谁
- **状态机**：当前处于哪一步
- **checkpoint**：中途断了能不能恢复

这三者组合起来，才有企业级执行能力。否则一旦进程挂掉，任务就得重来，成本会很高。

In [ ]:
workflow = {
    'intake': ['classify'],
    'classify': ['fetch_data', 'human_review'],
    'fetch_data': ['compute'],
    'compute': ['render', 'notify'],
    'human_review': ['notify'],
    'render': ['notify'],
    'notify': [],
}

def topo_order(graph):
    indeg = {n: 0 for n in graph}
    for src, targets in graph.items():
        for t in targets:
            indeg[t] = indeg.get(t, 0) + 1
    queue = [n for n, d in indeg.items() if d == 0]
    order = []
    while queue:
        node = queue.pop(0)
        order.append(node)
        for nxt in graph.get(node, []):
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                queue.append(nxt)
    return order

print('执行顺序:', ' -> '.join(topo_order(workflow)))

## 10. 业务映射：把编排能力放回 LangChat / OpenClaw / 商管 / 会员

### LangChat
- 把一个复杂需求拆成多个 Skill
- 每个 Skill 有自己的输入、输出、校验和审批
- 编排层决定 Skill 顺序和异常回退

### OpenClaw / Orchestrator
- 做入口路由、权限绑定、审计和追踪
- 把不同能力路由到不同后端
- 对异步事件、定时任务、外部消息做统一调度

### 商管 / 会员 / 物业
- 商管：月报、收费、合同、招商
- 会员：权益、积分、活动触达
- 物业：工单、巡检、能耗、告警

编排能力决定的是：这些系统能不能被 AI 串起来，而不是被 AI 各自孤立地问答。

In [ ]:
matrix = [
    ['场景', '推荐模式', '原因'],
    ['会员权益问答', '线性 + 分支', '规则明确，路径稳定'],
    ['经营月报生成', '并行 + Supervisor', '数据源多，适合分工'],
    ['工单超时提醒', '事件驱动', '靠事件触发最自然'],
    ['退款审批流', '状态机 + 人审', '高风险，需要 checkpoint'],
]
for row in matrix:
    print('	'.join(row))

## 11. 课堂练习

1. 把“会员升级咨询”拆成至少 4 步。
2. 选一个商管场景，写出你会在哪一步加人工确认。
3. 想一个会并行执行的任务，并说明为什么不能串行。
4. 把“每天 6:00 推送学习内容”写成一个 Cron 触发语义。

## 12. 课后测试

1. 为什么工作流更像图而不是链？
2. 线性流程最适合什么类型的任务？
3. Supervisor 模式的核心职责是什么？
4. Cron、延迟任务、周期任务分别适合什么场景？
5. checkpoint 在企业级执行里解决了什么问题？

In [ ]:
answers = {
    1: '因为现实任务经常分支、并行、回退，不是固定一条直线。',
    2: '确定性高、步骤稳定、分支少的任务。',
    3: '协调多个专员，决定谁做什么，并汇总结果。',
    4: 'Cron=固定时点；延迟=过一段时间；周期=固定间隔重复。',
    5: '解决中途失败后恢复执行，避免整单重跑。',
}
for k, v in answers.items():
    print(f'Q{k}: {v}')

## 13. 英文术语

| 术语 | 音标 | 含义 |
|---|---|---|
| Orchestration | /ˌɔːrkɪˈstreɪʃən/ | 编排，协调多个步骤或组件执行 |
| Workflow | /ˈwɜːrkfloʊ/ | 工作流，按规则执行的任务流程 |
| DAG | /dæɡ/ | 有向无环图，常用于依赖调度 |
| State Machine | /steɪt məˈʃiːn/ | 状态机，用状态和转移描述流程 |
| Supervisor | /ˌsuːpərˈvaɪzər/ | 主管节点，负责协调子任务 |
| Handoff | /ˈhændˌɔːf/ | 交接，把任务转给另一个专员 |
| Checkpoint | /ˈtʃekpɔɪnt/ | 检查点，保存中间状态便于恢复 |
| Idempotency | /ˌaɪdəmˈpoʊtənsi/ | 幂等性，重复执行也不会出错 |
| Event-driven | /ɪˈvent ˈdrɪvn/ | 事件驱动，事件触发系统响应 |
| Retry | /ˌriːˈtraɪ/ | 重试，失败后再次执行 |

## 14. 推荐资源

### 🎬 视频
1. [LangGraph Supervisor Agent: Multi-Agent Orchestration Walkthrough](https://www.youtube.com/watch?v=HonlBK19F1o)
2. [Build a Powerful Multi-Agent System Using LangGraph](https://www.youtube.com/watch?v=1SFZz8okqkg)

### 📖 文章
1. [LangChain Docs: Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
2. [LangChain Blog: LangGraph - Multi-Agent Workflows](https://www.langchain.com/blog/langgraph-multi-agent-workflows)

## 15. 今日总结

今天学的不是“更多概念”，而是把 Agent 变成真正能做事的运行时结构。

你可以把它记成一句话：
**行为设计解决“能不能做”，记忆系统解决“记不记得”，任务编排解决“能不能做完”。**

下一步，Day4 会进入多 Agent 协作模式：主 Agent、子 Agent、上下文传递和 TaskFlow。